# Sesion 2 - Metro Madrid: estacionalidad compleja y prediccion practica

Objetivo de la sesion:
- Modelar una serie real con fuerte estructura de calendario.
- Separar senal determinista (tendencia + estacionalidades) de ruido/incertidumbre.
- Comparar modelos por error puntual y por calidad de intervalos de confianza.

Conexion con la sesion 1:
- En FX (no estacionaria), el foco principal era rango de riesgo.
- En Metro, ademas de incertidumbre, buscamos capturar explicitamente estacionalidad semanal y anual.



## 1) Librerias

Base metodologica:
- `STL`: descomposicion interpretable de tendencia/estacionalidad/residuo.
- `SARIMAX`: modelo clasico para estructura temporal y estacional.
- `ExponentialSmoothing (ETS)`: baseline robusto para nivel/tendencia/estacionalidad.
- `Prophet`: alternativa muy usada en negocio para tendencia + multiples estacionalidades + changepoints.
- Metricas: `MAE`, `RMSE`, `Cobertura IC95`, `% fuera IC95`.



In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 4)



## 2) Carga y limpieza minima

Principio practico:
- Sin calidad de dato temporal, no hay modelo fiable.

Que hacemos y por que:
1. Parseo de fechas y orden temporal.
2. Conversion numerica de demanda.
3. Reindexado diario para garantizar regularidad temporal.
4. Imputacion simple de huecos (interpolacion temporal + arrastre) para no romper modelos.
5. Guardado de version limpia (`data/processed`) para trazabilidad.



In [ ]:
# Cargamos fichero original.
df = pd.read_csv("../data/raw/demanda_diaria_metro_2015_2024.csv")

# Parseamos fecha.
df["Fecha"] = pd.to_datetime(df["Fecha"])

# Renombramos para trabajar con nombres consistentes.
df = df.rename(columns={"Fecha": "date", "Demanda": "y"})

# Convertimos y a numerico y limpiamos nulos.
df["y"] = pd.to_numeric(df["y"], errors="coerce")
df = df.dropna(subset=["y"]).sort_values("date")

# Indice temporal diario.
df = df.set_index("date")

# Reindexamos a diario por robustez y rellenamos posibles huecos.
idx = pd.date_range(df.index.min(), df.index.max(), freq="D")
df = df.reindex(idx)
df["y"] = df["y"].interpolate(method="time").ffill().bfill()

print("Filas:", len(df))
print("Inicio:", df.index.min())
print("Fin:", df.index.max())
print("Nulos:", df["y"].isna().sum())



In [ ]:
# Guardamos limpio para trazabilidad.
df.reset_index().rename(columns={"index": "date"}).to_csv("../data/processed/metro_daily_clean.csv", index=False)



## 3) EDA de dominio

Lectura de negocio antes de modelar:
- Nivel global de demanda y su evolucion.
- Cambios de regimen (COVID).
- Patron semanal (laborables vs fin de semana).
- Patron anual (meses, vacaciones, navidad, etc.).

Esta fase define las hipotesis de modelado (que componente debe capturar el modelo y que parte queda como ruido).



In [ ]:
# Marcamos periodo COVID para analisis visual.
covid_start = pd.Timestamp("2020-03-01")
covid_end = pd.Timestamp("2022-12-31")
df["is_covid"] = (df.index >= covid_start) & (df.index <= covid_end)

# Serie completa con banda COVID.
plt.figure(figsize=(14, 4))
plt.plot(df.index, df["y"], color="tab:blue", linewidth=1)
plt.axvspan(covid_start, covid_end, color="red", alpha=0.12, label="Periodo COVID")
plt.title("Demanda diaria Metro Madrid")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend()
plt.show()

# Variables de calendario para analisis estacional.
df_eda = df.copy()
df_eda["dow"] = df_eda.index.day_name()
df_eda["month"] = df_eda.index.month

# Patron semanal.
order_dow = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
plt.figure(figsize=(10, 4))
sns.boxplot(data=df_eda, x="dow", y="y", order=order_dow)
plt.title("Distribucion por dia de semana")
plt.xlabel("Dia")
plt.ylabel("Viajeros")
plt.xticks(rotation=25)
plt.show()

# Patron anual por mes.
plt.figure(figsize=(10, 4))
sns.boxplot(data=df_eda, x="month", y="y")
plt.title("Distribucion por mes")
plt.xlabel("Mes")
plt.ylabel("Viajeros")
plt.show()



### 3.1) Zoom de un ano completo (detalle semanal + anual)

Finalidad didactica:
- Ver simultaneamente estructura semanal (ciclos cortos) y anual intra-ano.
- Detectar eventos y atipicos (festivos, verano, puentes).

Que mirar:
- Serie diaria (ruido + senal)
- Media movil 7 dias (estructura semanal)
- Media movil 30 dias (estructura de medio plazo)
- Distribucion de fines de semana frente a laborables



In [ ]:
# Elegimos el ultimo a?o completo disponible en la serie.
full_years = []
for y in sorted(df.index.year.unique()):
    i0 = pd.Timestamp(f"{y}-01-01")
    i1 = pd.Timestamp(f"{y}-12-31")
    if i0 in df.index and i1 in df.index:
        full_years.append(y)

if len(full_years) == 0:
    raise ValueError("No hay a?os completos en la serie para el zoom anual")

year_zoom = max(full_years)
start_y = pd.Timestamp(f"{year_zoom}-01-01")
end_y = pd.Timestamp(f"{year_zoom}-12-31")

df_year = df.loc[start_y:end_y].copy()

# Suavizados para leer mejor estructura semanal y anual.
df_year["ma_7"] = df_year["y"].rolling(7).mean()
df_year["ma_30"] = df_year["y"].rolling(30).mean()

# M?scara fin de semana para marcar patr?n semanal.
is_weekend = df_year.index.weekday >= 5

plt.figure(figsize=(14, 5))

# Serie diaria (ruidosa) en gris.
plt.plot(df_year.index, df_year["y"], color="lightgray", linewidth=1, label="Demanda diaria")

# Medias m?viles para estructura temporal.
plt.plot(df_year.index, df_year["ma_7"], color="tab:blue", linewidth=2, label="Media movil 7 dias")
plt.plot(df_year.index, df_year["ma_30"], color="tab:orange", linewidth=2, label="Media movil 30 dias")

# Marcamos fines de semana (patr?n semanal).
plt.scatter(df_year.index[is_weekend], df_year.loc[is_weekend, "y"], s=8, color="tab:red", alpha=0.35, label="Fin de semana")

# Marcamos verano para lectura de estacionalidad anual.
summer_start = pd.Timestamp(f"{year_zoom}-07-01")
summer_end = pd.Timestamp(f"{year_zoom}-08-31")
plt.axvspan(summer_start, summer_end, color="gold", alpha=0.15, label="Verano (jul-ago)")

plt.title(f"Zoom anual completo ({year_zoom}): estacionalidad semanal y anual")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend(loc="best")
plt.show()



## 4) Descomposicion: tendencia + estacionalidad anual + estacionalidad semanal + ruido

Identidad aditiva de trabajo:

`serie_observada = tendencia + estacionalidad_semanal + estacionalidad_anual + ruido`

Interpretacion:
- **Determinista**: tendencia + estacionalidades (parte explicable).
- **Aleatoria**: ruido/residuo (incertidumbre no explicada).

Notas tecnicas:
- `STL(period=7)` nos da una separacion robusta de componente semanal.
- La estacionalidad anual se aproxima con efecto mensual centrado (media cero) para no mezclar nivel con tendencia.
- El residuo es clave para calibrar intervalos de confianza.



In [ ]:
# STL semanal para separar tendencia/estacionalidad/residuo.
stl = STL(df["y"], period=7, robust=True).fit()

# Componentes STL.
df["trend_stl"] = stl.trend
df["seasonal_weekly_stl"] = stl.seasonal
df["resid_stl"] = stl.resid

# Estacionalidad anual explicita: efecto mensual centrado (media cero).
# 1) media por mes del a?o.
month_mean = df.groupby(df.index.month)["y"].mean()
# 2) nivel medio global.
global_mean = df["y"].mean()
# 3) efecto anual centrado.
month_effect_centered = month_mean - global_mean
# 4) asignamos el efecto a cada fila seg?n su mes.
df["seasonal_annual"] = df.index.month.map(month_effect_centered)

# Identidad aditiva completa.
df["deterministic_part"] = (
    df["trend_stl"]
    + df["seasonal_weekly_stl"]
    + df["seasonal_annual"]
)

df["noise_part"] = df["y"] - df["deterministic_part"]

# Verificacion numerica de la identidad (error medio absoluto de reconstruccion).
reconstruction_error = (
    df["y"]
    - (df["trend_stl"] + df["seasonal_weekly_stl"] + df["seasonal_annual"] + df["noise_part"])
).abs().mean()
print(f"Error medio absoluto de reconstruccion: {reconstruction_error:.6f}")

# Graficos de componentes.
fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)

axes[0].plot(df.index, df["y"], color="black")
axes[0].set_title("Serie observada")

axes[1].plot(df.index, df["trend_stl"], color="tab:blue")
axes[1].set_title("Tendencia")

axes[2].plot(df.index, df["seasonal_weekly_stl"], color="tab:green")
axes[2].set_title("Estacionalidad semanal (STL)")

axes[3].plot(df.index, df["seasonal_annual"], color="tab:orange")
axes[3].set_title("Estacionalidad anual (efecto mensual centrado)")

axes[4].plot(df.index, df["noise_part"], color="tab:red")
axes[4].set_title("Ruido / incertidumbre (residuo)")

plt.tight_layout()
plt.show()



In [ ]:
# Estadisticas del ruido para interpretar incertidumbre.
noise_std = df["noise_part"].std()
noise_mean = df["noise_part"].mean()

print(f"Media del ruido: {noise_mean:.2f}")
print(f"Desviacion tipica del ruido: {noise_std:.2f}")
print("Interpretacion: esta dispersion es la base para construir intervalos razonables.")



## 5) Forecast medio plazo: viajeros medios de los proximos 12 meses

Escenario practico:
- Planificacion de capacidad, presupuesto y operacion mensual.

Fundamento de modelos:
- **SARIMA mensual**: modela dependencia temporal + estacionalidad anual (periodo 12).
- **ETS mensual**: modela nivel/tendencia/estacionalidad de forma suavizada.
- **Prophet mensual**: modela tendencia con changepoints + estacionalidades de calendario + intervalo de incertidumbre.

Estrategia:
1. Agregar a serie mensual media.
2. Reservar ultimos 12 meses como pseudo-test.
3. Comparar modelos con:
   - Error puntual: MAE, RMSE
   - Incertidumbre: Cobertura IC95 y % fuera de intervalo
4. Reentrenar con todo el historico y proyectar proximos 12 meses.



In [ ]:
# Serie mensual (promedio de viajeros diarios por mes).
monthly = df["y"].resample("MS").mean()

# Pseudo-test: ultimos 12 meses.
train_m = monthly.iloc[:-12]
test_m = monthly.iloc[-12:]

# Modelo 1: SARIMA mensual con estacionalidad anual (12).
sarima_m = SARIMAX(train_m, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))
sarima_m_fit = sarima_m.fit(disp=False)
fc_m = sarima_m_fit.get_forecast(steps=12)
sarima_m_pred = fc_m.predicted_mean
sarima_m_ci = fc_m.conf_int(alpha=0.05)
sarima_m_lo = sarima_m_ci.iloc[:, 0]
sarima_m_hi = sarima_m_ci.iloc[:, 1]

# Modelo 2: Holt-Winters (ETS aditivo).
ets_m = ExponentialSmoothing(train_m, trend="add", seasonal="add", seasonal_periods=12).fit()
ets_m_pred = ets_m.forecast(12)
ets_sigma = pd.Series(ets_m.resid).std()
ets_m_lo = ets_m_pred - 1.96 * ets_sigma
ets_m_hi = ets_m_pred + 1.96 * ets_sigma

# Modelo 3: Prophet mensual (integrado, no bloque aparte).
prophet_monthly_available = False
try:
    from prophet import Prophet

    train_p_m = train_m.reset_index()
    train_p_m.columns = ["ds", "y"]

    model_p_m = Prophet(
        interval_width=0.95,
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
    )
    model_p_m.fit(train_p_m)

    future_p_m = model_p_m.make_future_dataframe(periods=12, freq="MS")
    fc_p_m = model_p_m.predict(future_p_m)

    fc_p_m_test = fc_p_m.set_index("ds").loc[test_m.index]
    prophet_m_pred = fc_p_m_test["yhat"]
    prophet_m_lo = fc_p_m_test["yhat_lower"]
    prophet_m_hi = fc_p_m_test["yhat_upper"]

    prophet_monthly_available = True
except Exception as e:
    print("Prophet mensual no disponible en esta ejecucion:", e)

# Metricas y cobertura.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rows_m = [
    {
        "Modelo": "SARIMA mensual",
        "MAE": mean_absolute_error(test_m, sarima_m_pred),
        "RMSE": rmse(test_m, sarima_m_pred),
        "Cobertura_IC95": ((test_m >= sarima_m_lo) & (test_m <= sarima_m_hi)).mean(),
    },
    {
        "Modelo": "ETS mensual",
        "MAE": mean_absolute_error(test_m, ets_m_pred),
        "RMSE": rmse(test_m, ets_m_pred),
        "Cobertura_IC95": ((test_m >= ets_m_lo) & (test_m <= ets_m_hi)).mean(),
    },
]

if prophet_monthly_available:
    rows_m.append(
        {
            "Modelo": "Prophet mensual",
            "MAE": mean_absolute_error(test_m, prophet_m_pred),
            "RMSE": rmse(test_m, prophet_m_pred),
            "Cobertura_IC95": ((test_m >= prophet_m_lo) & (test_m <= prophet_m_hi)).mean(),
        }
    )

summary_m = pd.DataFrame(rows_m)
summary_m["%Fuera_IC95"] = 1 - summary_m["Cobertura_IC95"]
summary_m = summary_m[["Modelo", "MAE", "RMSE", "Cobertura_IC95", "%Fuera_IC95"]]
summary_m = summary_m.sort_values("Modelo").reset_index(drop=True)
summary_m = summary_m.round({"MAE": 6, "RMSE": 6, "Cobertura_IC95": 6, "%Fuera_IC95": 6})
summary_m




In [ ]:
# Visual comparativa en pseudo-test mensual (una sola grafica).
plt.figure(figsize=(14, 4))
plt.plot(test_m.index, test_m.values, label="Real", color="black")

plt.plot(test_m.index, sarima_m_pred.values, label="SARIMA", color="tab:green")
plt.fill_between(test_m.index, sarima_m_lo.values, sarima_m_hi.values, color="tab:green", alpha=0.2, label="IC95 SARIMA")

plt.plot(test_m.index, ets_m_pred.values, label="ETS", color="tab:orange")
plt.fill_between(test_m.index, ets_m_lo.values, ets_m_hi.values, color="tab:orange", alpha=0.15, label="IC95 ETS")

if 'prophet_monthly_available' in globals() and prophet_monthly_available:
    plt.plot(test_m.index, prophet_m_pred.values, label="Prophet", color="tab:blue")
    plt.fill_between(test_m.index, prophet_m_lo.values, prophet_m_hi.values, color="tab:blue", alpha=0.12, label="IC95 Prophet")

plt.title("Validacion mensual (ultimos 12 meses)")
plt.xlabel("Fecha")
plt.ylabel("Viajeros medios diarios del mes")
plt.legend(loc="best")
plt.show()



In [ ]:
# Estadisticos de prediccion mensual (ademas de MAE/RMSE).
rows_stats_m = [
    {
        "Modelo": "SARIMA mensual",
        "Media_pred": sarima_m_pred.mean(),
        "Std_pred": sarima_m_pred.std(),
        "Media_intervalo_95": (sarima_m_hi - sarima_m_lo).mean(),
        "Min_pred": sarima_m_pred.min(),
        "Max_pred": sarima_m_pred.max(),
    },
    {
        "Modelo": "ETS mensual",
        "Media_pred": ets_m_pred.mean(),
        "Std_pred": ets_m_pred.std(),
        "Media_intervalo_95": (ets_m_hi - ets_m_lo).mean(),
        "Min_pred": ets_m_pred.min(),
        "Max_pred": ets_m_pred.max(),
    },
]

if 'prophet_monthly_available' in globals() and prophet_monthly_available:
    rows_stats_m.append(
        {
            "Modelo": "Prophet mensual",
            "Media_pred": prophet_m_pred.mean(),
            "Std_pred": prophet_m_pred.std(),
            "Media_intervalo_95": (prophet_m_hi - prophet_m_lo).mean(),
            "Min_pred": prophet_m_pred.min(),
            "Max_pred": prophet_m_pred.max(),
        }
    )

monthly_pred_stats = pd.DataFrame(rows_stats_m)
monthly_pred_stats



In [ ]:
# Forecast real: proximos 12 meses usando todo el historico.
monthly_full = monthly.copy()

# Reentrenamos SARIMA con toda la serie.
sarima_full = SARIMAX(monthly_full, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12)).fit(disp=False)
fc_12 = sarima_full.get_forecast(steps=12)
future_m = fc_12.predicted_mean
future_m_ci = fc_12.conf_int(alpha=0.05)

future_m_table = pd.DataFrame(
    {
        "y_pred": future_m.values,
        "y_lower_95": future_m_ci.iloc[:, 0].values,
        "y_upper_95": future_m_ci.iloc[:, 1].values,
    },
    index=future_m.index,
)
future_m_table



## 6) Forecast corto plazo: viajeros diarios del proximo mes

Escenario practico:
- Programacion operativa diaria (frecuencias, personal, incidencias).

Fundamento de modelos:
- **SARIMAX diario**: captura autocorrelacion y estacionalidad semanal explicita (periodo 7).
- **NHITS diario**: modelo deep learning especializado para patrones temporales no lineales.
- **Prophet diario**: alternativa de negocio con tendencia + estacionalidad semanal/anual + incertidumbre.

Estrategia:
1. Pseudo-test de ultimos 30 dias.
2. Comparar modelos con MAE, RMSE, cobertura IC95 y % fuera IC95.
3. Proyectar proximos 30 dias con intervalo de confianza.



In [ ]:
# Pseudo-test diario: ultimos 30 dias.
train_d = df["y"].iloc[:-30]
test_d = df["y"].iloc[-30:]

# Modelo base: naive estacional semanal (lag 7).
naive_d_pred = pd.concat([train_d, test_d]).shift(7).loc[test_d.index]
naive_sigma = (train_d - train_d.shift(7)).dropna().std()
naive_d_lo = naive_d_pred - 1.96 * naive_sigma
naive_d_hi = naive_d_pred + 1.96 * naive_sigma

# Modelo SARIMAX diario con estacionalidad semanal.
sarimax_d = SARIMAX(train_d, order=(1, 0, 1), seasonal_order=(1, 0, 1, 7), enforce_stationarity=False, enforce_invertibility=False)
sarimax_d_fit = sarimax_d.fit(disp=False)
fc_d = sarimax_d_fit.get_forecast(steps=30)
sarimax_d_pred = fc_d.predicted_mean
sarimax_d_ci = fc_d.conf_int(alpha=0.05)
sarimax_d_lo = sarimax_d_ci.iloc[:, 0]
sarimax_d_hi = sarimax_d_ci.iloc[:, 1]

# Prophet diario integrado.
prophet_daily_available = False
try:
    from prophet import Prophet

    train_p_d = train_d.reset_index()
    train_p_d.columns = ["ds", "y"]

    model_p_d = Prophet(
        interval_width=0.95,
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
    )
    model_p_d.fit(train_p_d)

    future_p_d = model_p_d.make_future_dataframe(periods=30, freq="D")
    fc_p_d = model_p_d.predict(future_p_d)

    fc_p_d_test = fc_p_d.set_index("ds").loc[test_d.index]
    prophet_d_pred = fc_p_d_test["yhat"]
    prophet_d_lo = fc_p_d_test["yhat_lower"]
    prophet_d_hi = fc_p_d_test["yhat_upper"]

    prophet_daily_available = True
except Exception as e:
    print("Prophet diario no disponible en esta ejecucion:", e)



### 6.1) Bonus sofisticado: NHITS (deep learning especializado)

Por que incluirlo:
- Es un modelo moderno para series temporales, util como contraste frente a modelos clasicos.

Criterio pedagogico:
- No asumir que "mas complejo" implica mejor.
- Validar igual que los modelos clasicos: MAE, RMSE y calidad de intervalos.



In [ ]:
# BONUS: NHITS en diario (mismo pseudo-test de 30 dias).
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from neuralforecast.losses.pytorch import MQLoss

train_nf = train_d.to_frame("y").reset_index().rename(columns={"index": "ds"})
train_nf["unique_id"] = "METRO"
train_nf = train_nf[["unique_id", "ds", "y"]]

nhits_d = NHITS(
    h=30,
    input_size=min(180, len(train_nf) - 1),
    max_steps=250,
    loss=MQLoss(level=[95]),
    random_seed=7,
)

nf_d = NeuralForecast(models=[nhits_d], freq="D")
nf_d.fit(df=train_nf)
fc_nh = nf_d.predict().reset_index().sort_values("ds")

# Nombres directos de salida NHITS.
nhits_d_pred = fc_nh["NHITS-median"].values
nhits_d_lo = fc_nh["NHITS-lo-95"].values
nhits_d_hi = fc_nh["NHITS-hi-95"].values

# Tabla comparativa diaria con todos los modelos disponibles.
rows_d = [
    {
        "Modelo": "Naive semanal",
        "MAE": mean_absolute_error(test_d, naive_d_pred),
        "RMSE": rmse(test_d, naive_d_pred),
        "Cobertura_IC95": ((test_d >= naive_d_lo) & (test_d <= naive_d_hi)).mean(),
    },
    {
        "Modelo": "SARIMAX diario",
        "MAE": mean_absolute_error(test_d, sarimax_d_pred),
        "RMSE": rmse(test_d, sarimax_d_pred),
        "Cobertura_IC95": ((test_d >= sarimax_d_lo) & (test_d <= sarimax_d_hi)).mean(),
    },
    {
        "Modelo": "NHITS diario",
        "MAE": mean_absolute_error(test_d.values, nhits_d_pred),
        "RMSE": rmse(test_d.values, nhits_d_pred),
        "Cobertura_IC95": ((test_d.values >= nhits_d_lo) & (test_d.values <= nhits_d_hi)).mean(),
    },
]

if 'prophet_daily_available' in globals() and prophet_daily_available:
    rows_d.append(
        {
            "Modelo": "Prophet diario",
            "MAE": mean_absolute_error(test_d, prophet_d_pred),
            "RMSE": rmse(test_d, prophet_d_pred),
            "Cobertura_IC95": ((test_d >= prophet_d_lo) & (test_d <= prophet_d_hi)).mean(),
        }
    )

summary_d = pd.DataFrame(rows_d)
summary_d["%Fuera_IC95"] = 1 - summary_d["Cobertura_IC95"]
summary_d = summary_d[["Modelo", "MAE", "RMSE", "Cobertura_IC95", "%Fuera_IC95"]]
summary_d = summary_d.sort_values("Modelo").reset_index(drop=True)
summary_d = summary_d.round({"MAE": 6, "RMSE": 6, "Cobertura_IC95": 6, "%Fuera_IC95": 6})
summary_d




In [ ]:
# Visual comparativa diaria en pseudo-test (30 dias) en una sola grafica.
plt.figure(figsize=(14, 4))
plt.plot(test_d.index, test_d.values, label="Real", color="black")

plt.plot(test_d.index, sarimax_d_pred.values, label="SARIMAX", color="tab:green")
plt.fill_between(test_d.index, sarimax_d_lo.values, sarimax_d_hi.values, color="tab:green", alpha=0.2, label="IC95 SARIMAX")

plt.plot(test_d.index, nhits_d_pred, label="NHITS", color="tab:purple")
plt.fill_between(test_d.index, nhits_d_lo, nhits_d_hi, color="tab:purple", alpha=0.15, label="IC95 NHITS")

if 'prophet_daily_available' in globals() and prophet_daily_available:
    plt.plot(test_d.index, prophet_d_pred.values, label="Prophet", color="tab:blue")
    plt.fill_between(test_d.index, prophet_d_lo.values, prophet_d_hi.values, color="tab:blue", alpha=0.12, label="IC95 Prophet")

plt.title("Validacion diaria (ultimos 30 dias)")
plt.xlabel("Fecha")
plt.ylabel("Viajeros")
plt.legend(loc="best")
plt.show()



In [ ]:
# Estadisticos de prediccion diaria.
rows_stats_d = [
    {
        "Modelo": "Naive semanal",
        "Media_pred": np.mean(naive_d_pred),
        "Std_pred": np.std(naive_d_pred),
        "Media_intervalo_95": np.mean(naive_d_hi - naive_d_lo),
        "Min_pred": np.min(naive_d_pred),
        "Max_pred": np.max(naive_d_pred),
    },
    {
        "Modelo": "SARIMAX diario",
        "Media_pred": np.mean(sarimax_d_pred),
        "Std_pred": np.std(sarimax_d_pred),
        "Media_intervalo_95": np.mean(sarimax_d_hi - sarimax_d_lo),
        "Min_pred": np.min(sarimax_d_pred),
        "Max_pred": np.max(sarimax_d_pred),
    },
    {
        "Modelo": "NHITS diario",
        "Media_pred": np.mean(nhits_d_pred),
        "Std_pred": np.std(nhits_d_pred),
        "Media_intervalo_95": np.mean(nhits_d_hi - nhits_d_lo),
        "Min_pred": np.min(nhits_d_pred),
        "Max_pred": np.max(nhits_d_pred),
    },
]

if 'prophet_daily_available' in globals() and prophet_daily_available:
    rows_stats_d.append(
        {
            "Modelo": "Prophet diario",
            "Media_pred": np.mean(prophet_d_pred),
            "Std_pred": np.std(prophet_d_pred),
            "Media_intervalo_95": np.mean(prophet_d_hi - prophet_d_lo),
            "Min_pred": np.min(prophet_d_pred),
            "Max_pred": np.max(prophet_d_pred),
        }
    )

daily_pred_stats = pd.DataFrame(rows_stats_d)
daily_pred_stats



In [ ]:
# Forecast real: proximos 30 dias con SARIMAX (practico operativo diario).
sarimax_full = SARIMAX(df["y"], order=(1, 0, 1), seasonal_order=(1, 0, 1, 7), enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
fc_next_30 = sarimax_full.get_forecast(steps=30)
next_30 = fc_next_30.predicted_mean
next_30_ci = fc_next_30.conf_int(alpha=0.05)

next_30_table = pd.DataFrame(
    {
        "y_pred": next_30.values,
        "y_lower_95": next_30_ci.iloc[:, 0].values,
        "y_upper_95": next_30_ci.iloc[:, 1].values,
    },
    index=next_30.index,
)
next_30_table.head(10)



## 7) Cierre conceptual

Mensajes clave:
1. La descomposicion ordena el problema: que parte es senal y que parte es incertidumbre.
2. La comparacion de modelos debe incluir siempre:
   - precision puntual (MAE/RMSE)
   - calibracion de incertidumbre (cobertura IC95).
3. En practica profesional, un modelo util no es solo el que acierta el punto, sino el que entrega rangos fiables para decidir.

